# Summarize Evaluation Results

This notebook loads completed evaluation results, aggregates scores by benchmark/category/supercategory, and generates Markdown and HTML reports.

## Workflow

1. Load result JSONs from `results/<model-name>/` directories
2. Build comparison table across models
3. Aggregate by category / supercategory (KMMLU, CLIcK)
4. Export to Markdown (`results/RESULTS.md`)
5. Generate standalone HTML report with charts

> **Prerequisite**: Run the "Save Results to JSON" cell in `1_kmcq_benchmark.ipynb` first to populate `results/<model>/`.

## Step 1: Configuration

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
RESULTS_DIR = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

Project root: /Users/hyochoi/dev/rhoai-lmeval-builder-lab
Results dir:  /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results
Exists: True


## Step 2: Load Results

Results are organized as `results/<model-name>/<job-name>.json`.
Each JSON file is in EvalHub format:

```json
{
  "model": {"name": "qwen3-14b", "url": "..."},
  "benchmarks": [{"id": "kmmlu", "metrics": {"overall_accuracy": 60}}]
}
```

In [2]:
def load_results(results_dir: Path) -> dict:
    """Load all result JSONs organized by model.
    Returns {model_name: {job_file_stem: data_dict}}.
    """
    all_results = {}
    if not results_dir.exists():
        print(f"Results directory not found: {results_dir}")
        return all_results

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        all_results[model_name] = {}
        for json_file in sorted(model_dir.glob("*.json")):
            try:
                with open(json_file) as f:
                    data = json.load(f)
                all_results[model_name][json_file.stem] = data
            except (json.JSONDecodeError, IOError) as e:
                print(f"  Skipped {json_file}: {e}")
    return all_results


all_results = load_results(RESULTS_DIR)
print(f"Loaded results for {len(all_results)} model(s):")
for model, jobs in all_results.items():
    print(f"  {model}: {len(jobs)} job(s)")

Loaded results for 1 model(s):
  qwen3-14b: 25 job(s)


## Step 3: Build Comparison Table

Extract `overall_accuracy` from each benchmark and build a `{task: {model: score}}` table.

In [3]:
def extract_benchmark_scores(data: dict) -> dict:
    """Extract {benchmark_id: overall_accuracy} from a single result file."""
    scores = {}
    for bm in data.get("benchmarks", []):
        bm_id = bm.get("id", "unknown")
        metrics = bm.get("metrics", {})
        acc = metrics.get("overall_accuracy")
        if acc is not None:
            scores[bm_id] = round(float(acc), 2)
    return scores


def build_score_table(all_results: dict) -> dict:
    """Build {benchmark: {model: best_score}}.
    If a model has multiple jobs for the same benchmark, keep the latest (last file).
    """
    table = {}
    for model_name, jobs in all_results.items():
        for job_file, data in jobs.items():
            for bm_id, score in extract_benchmark_scores(data).items():
                if bm_id not in table:
                    table[bm_id] = {}
                table[bm_id][model_name] = score
    return table


score_table = build_score_table(all_results)
print(f"Benchmarks found: {sorted(score_table.keys())}")
for bm, models in sorted(score_table.items()):
    scores_str = ", ".join(f"{m}={s}%" for m, s in sorted(models.items()))
    print(f"  {bm}: {scores_str}")

Benchmarks found: ['click', 'haerae', 'kmmlu', 'kmmlu_hard', 'kobest_boolq']
  click: qwen3-14b=66.82%
  haerae: qwen3-14b=54.64%
  kmmlu: qwen3-14b=48.3%
  kmmlu_hard: qwen3-14b=27.95%
  kobest_boolq: qwen3-14b=93.23%


## Step 4: Display Results Table

In [4]:
try:
    import pandas as pd

    if score_table:
        df = pd.DataFrame(score_table).T
        df.index.name = "Benchmark"
        df = df.sort_index()
        display(df.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No results to display. Run evaluations first.")
except ImportError:
    for bm, models in sorted(score_table.items()):
        print(f"\n{bm}:")
        for m, s in sorted(models.items(), key=lambda x: -x[1]):
            print(f"  {m}: {s:.2f}%")

,qwen3-14b
Benchmark,
click,66.82
haerae,54.64
kmmlu,48.30
kmmlu_hard,27.95
kobest_boolq,93.23


## Step 5: Category / Supercategory Aggregation

Korean benchmarks have hierarchical category structures:
- **KMMLU**: supercategory (STEM, HUMSS, Applied Science, Other) -> category (45 subjects)
- **CLIcK**: supercategory (Culture, Language) -> category (11 topics)

Category-level scores come from `category_accuracy.*` and `supercategory_accuracy.*` metrics.

In [5]:
def extract_category_scores(all_results: dict, benchmark_id: str, prefix: str = "category_accuracy.") -> dict:
    """Extract category-level scores for a specific benchmark.
    Returns {category: {model: score}}.
    """
    table = {}
    for model_name, jobs in all_results.items():
        for job_file, data in jobs.items():
            for bm in data.get("benchmarks", []):
                if bm.get("id") != benchmark_id:
                    continue
                metrics = bm.get("metrics", {})
                for key, value in metrics.items():
                    if key.startswith(prefix) and isinstance(value, (int, float)):
                        category = key[len(prefix):]
                        if category not in table:
                            table[category] = {}
                        table[category][model_name] = round(float(value), 2)
    return table


for bm_id in sorted(score_table.keys()):
    cat_scores = extract_category_scores(all_results, bm_id)
    super_scores = extract_category_scores(all_results, bm_id, prefix="supercategory_accuracy.")

    if super_scores:
        print(f"\n{bm_id} - Supercategory:")
        try:
            df_s = pd.DataFrame(super_scores).T.sort_index()
            df_s.index.name = "Supercategory"
            display(df_s.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
        except Exception:
            for cat, models in sorted(super_scores.items()):
                print(f"  {cat}: {models}")

    if cat_scores:
        print(f"\n{bm_id} - Category:")
        try:
            df_c = pd.DataFrame(cat_scores).T.sort_index()
            df_c.index.name = "Category"
            display(df_c.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
        except Exception:
            for cat, models in sorted(cat_scores.items()):
                print(f"  {cat}: {models}")


click - Supercategory:


,qwen3-14b
Supercategory,
Culture,65.65
Language,69.44



click - Category:


,qwen3-14b
Category,
Economy,81.36
Functional,82.35
Geography,71.20
Grammar,45.18
History,40.71
Law,56.16
Politics,77.38
Pop Culture,78.05
Society,80.91



haerae - Category:


...

,qwen3-14b
Supercategory,
Other,27.95



kmmlu_hard - Category:


,qwen3-14b
Category,
accounting,23.91
biology,23.00
chemistry,39.00
computer_science,36.00
criminal_law,28.00
ecology,23.00
electrical_engineering,19.00
electronics_engineering,43.00
gas_technology_and_engineering,24.00



kobest_boolq - Category:


,qwen3-14b
Category,
unknown,93.23


## Step 6: Export to Markdown

In [6]:
def score_table_to_markdown(score_table: dict, title: str = "Evaluation Results") -> str:
    """Convert score table to markdown format."""
    if not score_table:
        return "No results available."

    all_models = sorted(set(
        model for tasks in score_table.values() for model in tasks.keys()
    ))

    lines = [f"## {title}", ""]
    header = "| Benchmark | " + " | ".join(all_models) + " |"
    sep = "|:---" + "|---:" * len(all_models) + "|"
    lines.extend([header, sep])

    for bm in sorted(score_table.keys()):
        row = f"| {bm} |"
        for model in all_models:
            score = score_table[bm].get(model, "-")
            row += f" {score:.2f}% |" if isinstance(score, float) else f" {score} |"
        lines.append(row)

    return "\n".join(lines)


md_output = score_table_to_markdown(score_table, "Korean LLM Benchmark Results")
print(md_output)

md_path = RESULTS_DIR / "RESULTS.md"
md_path.write_text(md_output)
print(f"\nSaved to {md_path}")

## Korean LLM Benchmark Results

| Benchmark | qwen3-14b |
|:---|---:|
| click | 66.82% |
| haerae | 54.64% |
| kmmlu | 48.30% |
| kmmlu_hard | 27.95% |
| kobest_boolq | 93.23% |

Saved to /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results/RESULTS.md


## Step 7: Generate HTML Report

Use `generate_report.py` to produce a standalone HTML report with Chart.js visualizations.

In [7]:
!python generate_report.py --results-dir ../results --output ../results/report.html

print("\nOpen results/report.html in a browser to view the interactive report.")

Loading results from: /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results
Report generated: ../results/report.html
  Models: 1
  Tasks: 5



Open results/report.html in a browser to view the interactive report.
